# 🤖 Tit Chatbot & RAG Evaluation

Hệ thống đánh giá toàn diện cho **Tit Chatbot** - Agentic AI Assistant.

## Tổng quan Metrics

### Chatbot Metrics (6 metrics)
| # | Metric | Mô tả | Loại |
|---|--------|--------|------|
| 1 | **Correctness** | So sánh response với ground truth | LLM Judge |
| 2 | **Concision** | Độ ngắn gọn (response < 3x expected) | Heuristic |
| 3 | **Helpfulness** | Độ hữu ích tổng thể | LLM Judge |
| 4 | **Coherence** | Tính mạch lạc, logic | LLM Judge |
| 5 | **Safety** | Không chứa nội dung có hại | LLM Judge |
| 6 | **Latency** | Thời gian phản hồi < 30s | Heuristic |

### RAG Metrics (4 metrics gốc - giữ nguyên)
| # | Metric | Mô tả | Loại |
|---|--------|--------|------|
| 1 | **Correctness** | Response vs Reference Answer | LLM Judge |
| 2 | **Relevance** | Response vs Input Question | LLM Judge |
| 3 | **Groundedness** | Response vs Retrieved Documents | LLM Judge |
| 4 | **Retrieval Relevance** | Retrieved Docs vs Input Question | LLM Judge |

### Tool Selection Metric (1 metric)
| # | Metric | Mô tả | Loại |
|---|--------|--------|------|
| 1 | **Tool Selection** | Chọn đúng tool cho câu hỏi | LLM Judge |

**Judge LLM:** Groq (tương thích với Tit Chatbot stack, KHÔNG dùng OpenAI)

**Tracking:** LangSmith

---
## 1. Setup & Configuration

In [3]:
import sys
import os
from pathlib import Path

# Thêm project root và evaluations vào sys.path
EVAL_DIR = Path.cwd() if Path.cwd().name == "evaluations" else Path.cwd() / "evaluations"
PROJECT_ROOT = EVAL_DIR.parent

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(EVAL_DIR))

# Load environment
from dotenv import load_dotenv
import certifi

load_dotenv(PROJECT_ROOT / ".env")

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "tit"

print(f"Project root: {PROJECT_ROOT}")
print(f"Evaluations dir: {EVAL_DIR}")
print(f"GROQ_MODEL: {os.getenv('GROQ_MODEL')}")
print(f"LANGSMITH_PROJECT: {os.getenv('LANGSMITH_PROJECT')}")
print("✅ Setup complete!")

Project root: d:\Tit-Chatbot
Evaluations dir: d:\Tit-Chatbot\evaluations
GROQ_MODEL: openai/gpt-oss-20b
LANGSMITH_PROJECT: tit
✅ Setup complete!


In [4]:
from langsmith import Client

client = Client()
print("✅ LangSmith client connected!")

✅ LangSmith client connected!


---
## 2. Chatbot Evaluation

Đánh giá Tit Chatbot trên các câu hỏi tổng quát với 6 metrics:
- **Correctness** (LLM Judge)
- **Concision** (Heuristic)
- **Helpfulness** (LLM Judge)
- **Coherence** (LLM Judge)
- **Safety** (LLM Judge)
- **Latency** (Heuristic)

### 2.1 Tạo Chatbot Dataset

In [12]:
from eval_datasets import CHATBOT_DATASET_NAME, CHATBOT_EXAMPLES

# Tìm Dataset đã tồn tại trên LangSmith
existing_datasets = list(
    client.list_datasets(
        dataset_name=CHATBOT_DATASET_NAME
    )
)

if existing_datasets:
    # Tái sử dụng Dataset cũ
    dataset = existing_datasets[0]
    print(f"Using existing dataset: {dataset.name}")

    # Kiểm tra Dataset đã có examples chưa
    existing_examples = list(
        client.list_examples(dataset_id=dataset.id)
    )

    if not existing_examples:
        client.create_examples(
            dataset_id=dataset.id,
            examples=CHATBOT_EXAMPLES,
        )
        print("Added chatbot examples.")
    else:
        print(f"Dataset already has {len(existing_examples)} examples.")

else:
    # Chỉ tạo mới khi chưa tồn tại
    dataset = client.create_dataset(
        dataset_name=CHATBOT_DATASET_NAME
    )

    client.create_examples(
        dataset_id=dataset.id,
        examples=CHATBOT_EXAMPLES,
    )

    print("Created new chatbot dataset.")

Using existing dataset: Tit Chatbot Evaluation
Dataset already has 10 examples.


### 2.2 Định nghĩa Chatbot Metrics

In [13]:
from eval_metrics import (
    correctness, concision, helpfulness, coherence, safety, latency_check,
    CHATBOT_EVALUATORS,
)

print("📊 Chatbot Evaluators:")
for evaluator in CHATBOT_EVALUATORS:
    print(f"   • {evaluator.__name__}: {evaluator.__doc__.strip() if evaluator.__doc__ else ''}")

📊 Chatbot Evaluators:
   • correctness: RAG Metric 1: Đánh giá tính chính xác so với ground truth.
   • concision: Chatbot Metric 5: Kiểm tra câu trả lời có ngắn gọn không.
    Rule: response phải ngắn hơn 3x độ dài expected answer.
    Không cần LLM judge - pure heuristic.
   • helpfulness: Chatbot Metric 6: Đánh giá tính hữu ích của câu trả lời.
   • coherence: Chatbot Metric 7: Đánh giá tính mạch lạc của câu trả lời.
   • safety: Chatbot Metric 9: Kiểm tra tính an toàn của câu trả lời.
   • latency_check: Chatbot Metric 10: Kiểm tra thời gian phản hồi < 30 giây.
    Không cần LLM judge - pure heuristic.


### 2.3 Target Function - Gọi Tit Chatbot thật

In [5]:
import time
from langchain_core.messages import HumanMessage, AIMessage
from agent import get_agent


def chatbot_target(inputs: dict) -> dict:
    """
    Gọi Tit Chatbot agent thực tế.
    Trả về response tương thích với tất cả metrics.
    """
    question = inputs.get("question", "")
    if not question.strip():
        return {
            "response": "Please provide a question.",
            "answer": "Please provide a question.",
            "latency_seconds": 0.0,
        }

    agent = get_agent()  # Dùng default model
    config = {"configurable": {"thread_id": f"eval-{hash(question) % 100000}"}}

    start = time.time()
    try:
        result = agent.invoke(
            {"messages": [HumanMessage(content=question)]},
            config=config,
        )
        elapsed = time.time() - start

        # Lấy AI response cuối cùng (bỏ qua tool calls)
        ai_msgs = [
            m for m in result["messages"]
            if isinstance(m, AIMessage) and m.content and not m.tool_calls
        ]
        response = ai_msgs[-1].content if ai_msgs else ""

        return {
            "response": response,
            "answer": response,
            "latency_seconds": round(elapsed, 3),
        }
    except Exception as e:
        return {
            "response": f"Error: {e}",
            "answer": f"Error: {e}",
            "latency_seconds": round(time.time() - start, 3),
        }


# Test thử
test = chatbot_target({"question": "What is Python?"})
print(f"Response: {test['answer'][:100]}...")
print(f"Latency: {test['latency_seconds']}s")

c:\Users\Admin\miniconda3\envs\tit\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2181.55it/s]


Response: Python is a high‑level, interpreted programming language known for its readability and versatility. ...
Latency: 1.213s


### 2.4 Chạy Chatbot Evaluation

In [14]:
chatbot_results = client.evaluate(
    chatbot_target,
    data=CHATBOT_DATASET_NAME,
    evaluators=CHATBOT_EVALUATORS,
    experiment_prefix="tit-chatbot",
    metadata={"model": os.getenv("GROQ_MODEL"), "type": "chatbot"},
)

print("✅ Chatbot evaluation complete!")

View the evaluation results for experiment: 'tit-chatbot-263725fb' at:
https://smith.langchain.com/o/f2e83ad4-604d-4b9e-81a8-9e78f389bf76/datasets/90dbad43-0456-42fb-bad7-4267657feeaf/compare?selectedSessions=1082764c-07bd-4c23-9b6b-a9498fc87422




1it [00:04,  4.91s/it]Error running evaluator <DynamicRunEvaluator coherence> on run 01a19fd0-9459-7476-82cf-020dacaccac5: RateLimitError("Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m5rpb57seahtwfw9mgfjbrhn` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6786, Requested 1442. Please try again in 1.71s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}")
Traceback (most recent call last):
  File "c:\Users\Admin\miniconda3\envs\tit\Lib\site-packages\langsmith\evaluation\_runner.py", line 1629, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Admin\miniconda3\envs\tit\Lib\site-packages\langsmith\evaluation\evaluator.py", line 353, in evaluate_run
    result = self.func(
  

✅ Chatbot evaluation complete!


In [10]:
# Xem kết quả dạng DataFrame
chatbot_results.to_pandas()

,inputs.question,outputs.response,outputs.answer,outputs.latency_seconds,error,reference.answer,feedback.correctness,feedback.concision,feedback.helpfulness,feedback.coherence,feedback.safety,feedback.latency_check,execution_time,example_id,id
0,Explain what a REST API is.,A **REST API** (Representational State Transfe...,A **REST API** (Representational State Transfe...,2.038,None,A REST API is an application programming inter...,True,False,True,True,True,True,2.040252,120b74d4-3287-415b-94d4-0fe403401edb,01a19c92-9cfd-7c92-aa99-3e3fd8004cbf
1,What is FastAPI?,"FastAPI is a modern, fast (high‑performance) w...","FastAPI is a modern, fast (high‑performance) w...",1.455,None,"FastAPI is a modern, fast web framework for bu...",True,False,True,True,True,True,1.454995,1ebd4af9-2b5a-4713-bfbf-984a9286e484,01a19c92-b7ee-7d92-a6b5-0ea4d605835c
2,Tell me a joke.,Why don’t skeletons fight each other?\n\nThey ...,Why don’t skeletons fight each other?\n\nThey ...,3.127,None,A humorous response related to programming or ...,False,True,True,True,True,True,3.127085,3105b627-79fd-40d8-8565-41f9405c9c65,01a19c92-f23a-7ec6-97ef-e2fcb34c63f4
3,What is machine learning?,Machine learning (ML) is a branch of artificia...,Machine learning (ML) is a branch of artificia...,6.349,None,Machine learning is a subset of artificial int...,True,False,True,True,True,True,6.348984,33619d79-a7c0-4449-bd5d-f0337bf035c3,01a19c93-4309-7fd4-bd13-d174e0f4b5b1
4,Remember that my favorite programming language...,Error: (sqlite3.OperationalError) no such tabl...,Error: (sqlite3.OperationalError) no such tabl...,6.475,None,I've saved that your favorite programming lang...,True,False,False,True,None,True,6.474921,4500cf51-67d5-4c2f-88b0-b7c83a1bb21d,01a19c93-d3d7-7cc9-8f40-f0d98be81029
5,What is Docker?,Docker is an open‑source platform that simplif...,Docker is an open‑source platform that simplif...,3.619,None,"Docker is a platform for developing, shipping,...",True,False,True,True,True,True,3.619417,4a59e8af-76c6-462c-b7bc-c4e6b92e57d6,01a19c94-3a83-7f02-91d0-6899d0db294a
6,,Please provide a question.,Please provide a question.,0.000,None,Please provide a question or message so I can ...,False,True,False,False,None,True,0.001509,d1957dc7-81a2-459f-bb89-48b2a6d40211,01a19c94-aa06-7035-a382-e79ff1538c19
7,Calculate the square root of 144.,The square root of 144 is **12**.,The square root of 144 is **12**.,21.484,None,12,True,False,True,True,True,True,21.484692,e45bb9d5-87e9-4d21-8d58-d16b38949c3f,01a19c94-f0dc-7dda-be6d-b006b75bc026
8,What is 15 * 23?,15 × 23 = **345**.,15 × 23 = **345**.,11.927,None,345,True,False,True,True,True,True,11.927293,f434c87c-a07b-4597-9136-4046c428589a,01a19c95-7bd1-7547-bd9a-30703a8d0fde
9,What is Python?,"Python is a high‑level, interpreted programmin...","Python is a high‑level, interpreted programmin...",7.956,None,"Python is a high-level, interpreted programmin...",True,False,True,True,True,True,7.956628,fe3a1d85-f6d5-4721-9985-debb5fb1a957,01a19c95-e710-74b1-b1bf-1b602b830fb6


---
## 3. Model Comparison

So sánh hiệu suất giữa các models Groq trên cùng dataset.

In [ ]:
from eval_config import CHATBOT_MODELS

print(f"Models to compare: {CHATBOT_MODELS}")

comparison_results = {}

for model_name in CHATBOT_MODELS:
    prefix = model_name.replace("/", "-")
    print(f"\n🔄 Evaluating: {model_name}")

    def make_target(m):
        def target(inputs: dict) -> dict:
            question = inputs.get("question", "")
            if not question.strip():
                return {"response": "", "answer": "", "latency_seconds": 0.0}

            agent = get_agent(m)
            config = {"configurable": {"thread_id": f"eval-cmp-{hash(question) % 100000}"}}

            start = time.time()
            result = agent.invoke(
                {"messages": [HumanMessage(content=question)]},
                config=config,
            )
            elapsed = time.time() - start

            ai_msgs = [
                msg for msg in result["messages"]
                if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls
            ]
            response = ai_msgs[-1].content if ai_msgs else ""

            return {
                "response": response,
                "answer": response,
                "latency_seconds": round(elapsed, 3),
            }
        return target

    results = client.evaluate(
        make_target(model_name),
        data=CHATBOT_DATASET_NAME,
        evaluators=CHATBOT_EVALUATORS,
        experiment_prefix=f"tit-{prefix}",
        metadata={"model": model_name, "type": "model_comparison"},
    )
    comparison_results[model_name] = results
    print(f"   ✅ Done: {model_name}")

print("\n📊 Check LangSmith for side-by-side comparison!")

Models to compare: ['openai/gpt-oss-20b', 'qwen/qwen3.8-27b']

🔄 Evaluating: openai/gpt-oss-20b
View the evaluation results for experiment: 'tit-openai-gpt-oss-20b-e5d7d229' at:
https://smith.langchain.com/o/f2e83ad4-604d-4b9e-81a8-9e78f389bf76/datasets/90dbad43-0456-42fb-bad7-4267657feeaf/compare?selectedSessions=60166a39-8acd-43cc-aa55-60bda8d168c6




0it [00:00, ?it/s]Error running target function: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m5rpb57seahtwfw9mgfjbrhn` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198224, Requested 4221. Please try again in 17m36.24s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Traceback (most recent call last):
  File "c:\Users\Admin\miniconda3\envs\tit\Lib\site-packages\langsmith\evaluation\_runner.py", line 1949, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "c:\Users\Admin\miniconda3\envs\tit\Lib\site-packages\langsmith\run_helpers.py", line 798, in wrapper
    function_result = run_container["context"].run(func, *args, **kwargs)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_25400\3468255740.py", line 21, in tar

   ✅ Done: openai/gpt-oss-20b

🔄 Evaluating: qwen/qwen3.8-27b
View the evaluation results for experiment: 'tit-qwen-qwen3.8-27b-0bf65282' at:
https://smith.langchain.com/o/f2e83ad4-604d-4b9e-81a8-9e78f389bf76/datasets/90dbad43-0456-42fb-bad7-4267657feeaf/compare?selectedSessions=65f39a5d-59ba-4aa7-90e0-78749a67d25d




0it [00:00, ?it/s]Error running evaluator <DynamicRunEvaluator correctness> on run 01a19cbb-9add-7c39-8218-2b1e70690915: RateLimitError("Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m5rpb57seahtwfw9mgfjbrhn` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199861, Requested 1190. Please try again in 7m34.032s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}")
Traceback (most recent call last):
  File "c:\Users\Admin\miniconda3\envs\tit\Lib\site-packages\langsmith\evaluation\_runner.py", line 1629, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Admin\miniconda3\envs\tit\Lib\site-packages\langsmith\evaluation\evaluator.py", line 353, in evaluate_run
    result = self.func(

---
## 4. Tool Selection Evaluation

Kiểm tra chatbot có chọn đúng tool cho từng loại câu hỏi:
- `calculator` cho math
- `tavily_search` cho tin tức/thời sự
- `search_uploaded_documents` cho document queries
- `remember_this` cho lưu nhớ
- `recall_memory` cho nhắc lại
- `none` cho kiến thức chung

In [4]:
from database import init_db, engine
from sqlalchemy import inspect

# Tạo các bảng nếu chưa tồn tại
init_db()

# Kiểm tra
print("Database:", engine.url.database)
print("Tables:", inspect(engine).get_table_names())

Database: D:/Tit-Chatbot/data/chatbot_memory.db
Tables: ['chat_messages', 'conversations', 'long_term_memory']


In [8]:
from eval_datasets import (
    TOOL_SELECTION_DATASET_NAME,
    TOOL_SELECTION_EXAMPLES,
)

from eval_metrics import tool_selection, TOOL_EVALUATORS

# Kiểm tra Dataset đã tồn tại chưa
existing_datasets = list(
    client.list_datasets(
        dataset_name=TOOL_SELECTION_DATASET_NAME
    )
)

if existing_datasets:
    # Tái sử dụng Dataset cũ
    tool_dataset = existing_datasets[0]
    print("Using existing dataset:", tool_dataset.name)

else:
    # Chỉ tạo Dataset khi chưa tồn tại
    tool_dataset = client.create_dataset(
        TOOL_SELECTION_DATASET_NAME
    )

    client.create_examples(
        dataset_id=tool_dataset.id,
        examples=TOOL_SELECTION_EXAMPLES,
    )

    print("Created new dataset:", tool_dataset.name)


# Kiểm tra số lượng examples
existing_examples = list(
    client.list_examples(dataset_id=tool_dataset.id)
)

print("Number of examples:", len(existing_examples))

Using existing dataset: Tit Tool Selection Evaluation
Number of examples: 8


In [6]:
from langchain_core.messages import ToolMessage


def tool_target(inputs: dict) -> dict:
    """Gọi Tit Chatbot và theo dõi tool nào được gọi."""
    question = inputs.get("question", "")
    agent = get_agent()
    config = {"configurable": {"thread_id": f"eval-tool-{hash(question) % 100000}"}}

    try:
        result = agent.invoke(
            {"messages": [HumanMessage(content=question)]},
            config=config,
        )

        tools_used = []
        for msg in result["messages"]:
            if isinstance(msg, AIMessage) and msg.tool_calls:
                for tc in msg.tool_calls:
                    tools_used.append(tc["name"])

        return {
            "tools_used": tools_used if tools_used else ["none"],
            "response": result["messages"][-1].content if result["messages"] else "",
        }
    except Exception as e:
        return {"tools_used": ["error"], "response": str(e)}


# Test
test = tool_target({"question": "What is 2 + 2?"})
print(f"Tools used: {test['tools_used']}")
print(f"Response: {test['response'][:100]}")

Tools used: ['calculator']
Response: The result of 2 + 2 is **4**.


In [9]:
tool_results = client.evaluate(
    tool_target,
    data=TOOL_SELECTION_DATASET_NAME,
    evaluators=TOOL_EVALUATORS,
    experiment_prefix="tit-tool-selection-fixed",
    metadata={"type": "tool_selection"},
)

print("✅ Tool selection evaluation complete!")
tool_results.to_pandas()

View the evaluation results for experiment: 'tit-tool-selection-fixed-0dffe878' at:
https://smith.langchain.com/o/f2e83ad4-604d-4b9e-81a8-9e78f389bf76/datasets/934aaa94-d287-4242-82d0-df7a661fd4b9/compare?selectedSessions=f66b41d1-c09a-4910-aa57-892032f6d0d5




8it [01:49, 13.67s/it]

✅ Tool selection evaluation complete!


,inputs.question,outputs.tools_used,outputs.response,error,reference.expected_tool,feedback.tool_selection,execution_time,example_id,id
0,Calculate math.sqrt(256) + 10,[calculator],The result is **26**.,None,calculator,True,1.474422,18c75dde-cb28-4860-aab8-8d9ecdeda92d,01a19fcb-cd02-7dfc-b643-8c86306607a0
1,What did I tell you to remember?,[recall_memory],You asked me to remember that your name is **S...,None,recall_memory,True,1.386484,50d772a9-2eba-46a8-a909-bb8488e432ab,01a19fcb-d5f4-7abc-b74a-ebfbb1101517
2,What is 2 + 2?,[calculator],2 + 2 equals **4**.,None,calculator,True,0.402535,74923604-9f5c-4b08-8d3e-9bc649533d78,01a19fcb-de51-77cb-9903-1655db6ffb68
3,Remember that I like coffee.,[remember_this],Got it! I’ve noted that you like coffee. Let m...,None,remember_this,True,2.818379,bc669e01-c329-4f91-a8df-47f207fe6faa,01a19fcb-e325-7743-92df-e17d1bd13307
4,What is the capital of France?,[none],The capital of France is **Paris**.,None,none,True,0.622671,c298f552-8558-4cfb-b802-e855688e1bb5,01a19fcb-f14a-73af-81e6-aa6298886e27
5,Search my uploaded document about machine lear...,[search_uploaded_documents],I couldn’t find any uploaded document that mat...,None,search_uploaded_documents,True,12.222923,c3534cc2-5916-47eb-95c9-ad6e20c4b3f5,01a19fcb-ffa7-7d4c-8bb5-9042bdcca112
6,What is the current price of Bitcoin?,[tavily_search],The most recent data from reputable crypto‑pri...,None,tavily_search,True,32.844434,d655acca-bf17-42ed-90ab-24e3e513cd0c,01a19fcc-42cf-7b91-a61d-953a659e9fee
7,What is the latest news about AI?,[tavily_search],**Latest AI news (as of Oct 2026)** – summary ...,None,tavily_search,True,34.765349,dd208c27-f4fb-4c0b-b4ea-183d15d32d9a,01a19fcc-d65e-757a-9920-87bbde694bb4


---
## 5. RAG Evaluation

Đánh giá RAG pipeline với **4 metrics gốc** (giữ nguyên):
1. **Correctness** - Response vs Reference Answer
2. **Relevance** - Response vs Input Question
3. **Groundedness** - Response vs Retrieved Documents
4. **Retrieval Relevance** - Retrieved Docs vs Input Question

### 5.1 Setup: Upload Test Document vào RAG Pipeline

In [15]:
from eval_datasets import RAG_DATASET_NAME, RAG_EXAMPLES, RAG_TEST_DOCUMENT
from eval_config import RAG_THREAD_ID
from rag import add_document_to_rag, vectorstore

# Tạo test document file
test_doc_path = PROJECT_ROOT / "uploads" / "eval_test_document.txt"
test_doc_path.write_text(RAG_TEST_DOCUMENT, encoding="utf-8")

# Upload vào RAG pipeline (giống hệt flow thật)
result = add_document_to_rag(
    file_path=str(test_doc_path),
    thread_id=RAG_THREAD_ID,
)

print(f"✅ Uploaded: {result['filename']} → {result['chunks']} chunks")

✅ Uploaded: eval_test_document.txt → 2 chunks


In [16]:
# Test retrieval
test_docs = vectorstore.similarity_search(
    "What is deep learning?",
    k=4,
    filter={"thread_id": RAG_THREAD_ID},
)

print(f"Retrieved {len(test_docs)} documents:")
for i, doc in enumerate(test_docs):
    print(f"  {i+1}. {doc.page_content[:80]}...")

Retrieved 2 documents:
  1. Artificial Intelligence (AI) is a branch of computer science that aims to create...
  2. Natural Language Processing (NLP) is a field of AI that gives machines the abili...


### 5.2 Tạo RAG Dataset

In [17]:
rag_dataset = client.create_dataset(RAG_DATASET_NAME)
client.create_examples(
    dataset_id=rag_dataset.id,
    examples=RAG_EXAMPLES,
)

print(f"✅ Created: {RAG_DATASET_NAME} ({len(RAG_EXAMPLES)} examples)")
for ex in RAG_EXAMPLES:
    print(f"   Q: {ex['inputs']['question']}")

✅ Created: Tit RAG Evaluation (5 examples)
   Q: What is Deep Learning?
   Q: What are the three types of learning in AI?
   Q: What is RAG and why is it useful?
   Q: What NLP tasks are mentioned in the document?
   Q: What does LangChain provide?


### 5.3 Định nghĩa RAG Metrics (4 metrics gốc)

In [18]:
from eval_metrics import (
    correctness as rag_correctness,
    relevance, groundedness, retrieval_relevance,
    RAG_EVALUATORS,
)

print("📊 RAG Evaluators:")
for evaluator in RAG_EVALUATORS:
    print(f"   • {evaluator.__name__}: {evaluator.__doc__.strip() if evaluator.__doc__ else ''}")

📊 RAG Evaluators:
   • correctness: RAG Metric 1: Đánh giá tính chính xác so với ground truth.
   • relevance: RAG Metric 2: Đánh giá tính liên quan của câu trả lời với câu hỏi.
   • groundedness: RAG Metric 3: Đánh giá câu trả lời có dựa trên documents hay không.
   • retrieval_relevance: RAG Metric 4: Đánh giá documents retrieved có liên quan đến câu hỏi.


### 5.4 RAG Target Function

In [19]:
from langsmith import traceable
from langchain_groq import ChatGroq


@traceable()
def rag_target(inputs: dict) -> dict:
    """
    RAG pipeline của Tit Chatbot:
      1. Retrieve documents từ ChromaDB
      2. Generate answer bằng Groq LLM
      3. Trả về {answer, documents}
    """
    question = inputs["question"]

    # Step 1: Retrieve
    docs = vectorstore.similarity_search(
        question, k=4,
        filter={"thread_id": RAG_THREAD_ID},
    )

    if not docs:
        return {"answer": "No relevant documents found.", "documents": []}

    # Step 2: Generate
    docs_string = "\n\n".join(doc.page_content for doc in docs)
    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions. \
Use the following source documents to answer the user's questions. \
If you don't know the answer, just say that you don't know. \
Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""

    llm = ChatGroq(model=os.getenv("GROQ_MODEL", "openai/gpt-oss-20b"), temperature=0)
    ai_msg = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ])

    return {"answer": ai_msg.content, "documents": docs}


# Test
test = rag_target({"question": "What is Deep Learning?"})
print(f"Answer: {test['answer'][:150]}...")
print(f"Documents retrieved: {len(test['documents'])}")

Answer: Deep Learning is a subset of machine learning that uses neural networks with many layers, called deep neural networks. It excels at tasks such as imag...
Documents retrieved: 2


### 5.5 Chạy RAG Evaluation

In [20]:
rag_results = client.evaluate(
    rag_target,
    data=RAG_DATASET_NAME,
    evaluators=RAG_EVALUATORS,
    experiment_prefix="tit-rag",
    metadata={
        "type": "rag",
        "version": "HuggingFace Embeddings + ChromaDB",
    },
)

print("✅ RAG evaluation complete!")

View the evaluation results for experiment: 'tit-rag-db4686b3' at:
https://smith.langchain.com/o/f2e83ad4-604d-4b9e-81a8-9e78f389bf76/datasets/8375a969-a0bf-474f-97dc-513d8994ced5/compare?selectedSessions=09f6496a-501b-4dd4-ab04-0016cd1cadca




5it [02:19, 27.94s/it]

✅ RAG evaluation complete!


In [21]:
# Xem kết quả dạng DataFrame
rag_results.to_pandas()

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.relevance,feedback.groundedness,feedback.retrieval_relevance,execution_time,example_id,id
0,What is RAG and why is it useful?,"RAG, or Retrieval‑Augmented Generation, is a t...",[page_content='Natural Language Processing (NL...,None,RAG (Retrieval Augmented Generation) combines ...,True,True,True,True,1.178218,2271b65d-0813-4368-a85f-6cec19d98238,01a19ca4-1d64-7501-a830-6f84ababfe2d
1,What NLP tasks are mentioned in the document?,The document lists five key NLP tasks: text cl...,[page_content='Natural Language Processing (NL...,None,The NLP tasks mentioned include text classific...,True,True,True,True,3.984924,355ea7cb-7740-445e-aab9-ac1ee9c297cc,01a19ca4-92c0-73cb-989e-a58c57ee5fe7
2,What is Deep Learning?,Deep Learning is a subset of machine learning ...,[page_content='Artificial Intelligence (AI) is...,None,Deep Learning is a subset of machine learning ...,True,True,True,True,6.140805,423d75a5-5b36-4b07-84db-8582fcf4aa6e,01a19ca4-f480-7591-9405-38a32d412a3f
3,What are the three types of learning in AI?,The three types of learning in AI are:\n\n1. *...,[page_content='Artificial Intelligence (AI) is...,None,The three types of learning in AI are: (1) Sup...,True,True,True,True,3.008445,b1e81d2d-d069-4714-b113-56938215baf8,01a19ca5-6623-758c-97b4-2d723605f1ab
4,What does LangChain provide?,LangChain is a framework for building applicat...,[page_content='Natural Language Processing (NL...,None,LangChain provides tools for prompt management...,True,True,True,True,5.921187,e0327b36-2e57-46bd-a977-de4041dd2a79,01a19ca5-c9a1-74b5-a0a2-dd7aa8f9e8f7


---
## 6. Cleanup (Optional)

Xóa test data khỏi ChromaDB sau khi đánh giá xong.

In [22]:
# Xóa test documents từ ChromaDB
collection = vectorstore._collection
results = collection.get(where={"thread_id": RAG_THREAD_ID})

if results and results["ids"]:
    collection.delete(ids=results["ids"])
    print(f"✅ Deleted {len(results['ids'])} test chunks from ChromaDB")

# Xóa test file
if test_doc_path.exists():
    test_doc_path.unlink()
    print(f"✅ Deleted: {test_doc_path.name}")

print("🧹 Cleanup complete!")

✅ Deleted 2 test chunks from ChromaDB
✅ Deleted: eval_test_document.txt
🧹 Cleanup complete!


### DEBUG TOOL REMEBER_THIS & RECALL_THIS

In [11]:
import database
import tools
from pathlib import Path
from sqlalchemy import inspect

print("Notebook directory:", Path.cwd())

print("\nDatabase module:")
print(database.__file__)

print("\nTools module:")
print(tools.__file__)

print("\nDatabase URL:")
print(database.engine.url)

print("\nActual database path:")
print(Path(database.engine.url.database).resolve())

print("\nModels registered:")
print(list(database.Base.metadata.tables.keys()))

print("\nTables before init:")
print(inspect(database.engine).get_table_names())

Notebook directory: d:\Tit-Chatbot\evaluations

Database module:
d:\Tit-Chatbot\database.py

Tools module:
D:\Tit-Chatbot\tools.py

Database URL:
sqlite:///D:/Tit-Chatbot/data/chatbot_memory.db

Actual database path:
D:\Tit-Chatbot\data\chatbot_memory.db

Models registered:
['conversations', 'chat_messages', 'long_term_memory']

Tables before init:
['chat_messages', 'conversations', 'long_term_memory']


In [12]:
database.init_db()

print("Tables after init:")
print(inspect(database.engine).get_table_names())

Tables after init:
['chat_messages', 'conversations', 'long_term_memory']


In [13]:
print("Save memory module:")
print(tools.save_memory.__module__)

print("Database used by save_memory:")
print(tools.save_memory.__globals__["SessionLocal"].kw["bind"].url)

print("Database used by search_memory:")
print(tools.search_memory.__globals__["SessionLocal"].kw["bind"].url)

Save memory module:
database
Database used by save_memory:
sqlite:///D:/Tit-Chatbot/data/chatbot_memory.db
Database used by search_memory:
sqlite:///D:/Tit-Chatbot/data/chatbot_memory.db


In [14]:
database.save_memory(
    thread_id="debug-test",
    memory="I like coffee."
)

print(
    database.search_memory(
        thread_id="debug-test",
        query="coffee"
    )
)

- I like coffee.


In [15]:
import tools

# Tạo thread riêng để test
tools.set_current_thread_id("eval-memory-debug")

# Test remember_this
print(
    tools.remember_this.invoke({
        "memory": "I like coffee."
    })
)

# Test recall_memory
print(
    tools.recall_memory.invoke({
        "query": "coffee"
    })
)

Memory saved successfully.
- I like coffee.


---
## 📊 Summary

| Evaluation | Metrics | Dataset |
|---|---|---|
| Chatbot | correctness, concision, helpfulness, coherence, safety, latency | Tit Chatbot Evaluation |
| Model Compare | same as chatbot, across multiple models | Tit Chatbot Evaluation |
| Tool Selection | tool_selection | Tit Tool Selection Evaluation |
| RAG | correctness, relevance, groundedness, retrieval_relevance | Tit RAG Evaluation |

📈 **Xem kết quả chi tiết tại:** [LangSmith Dashboard](https://smith.langchain.com)